# MOTCO End-to-End Example

This notebook walks through the complete MOTCO workflow using the bundled toy dataset at `data/toy/`.

**Pipeline overview:**
1. [Data generation](#1-Data-generation) — how the toy data was created
2. [PLS-DA latent space](#2-PLS-DA-latent-space) — supervised integration (X = omics, y = disease stage)
3. [SNF latent space](#3-SNF-latent-space) — unsupervised integration (alternative path)
4. [Trajectory design](#4-Trajectory-design) — load pre-generated model matrices
5. [Differential trajectory analysis](#5-Differential-trajectory-analysis) — estimate differences + RRPP
6. [Visualization](#6-Visualization) — plot group trajectories

**Toy dataset:** 90 samples across 3 disease stages (0 → 1 → 2), two groups (A and B), three omics layers (methylation, expression, proteomics). Group B's trajectory is rotated ~90° relative to group A (orientation-mode effect), making this an ideal case to showcase MOTCO.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TOY = Path('data/toy')
assert TOY.exists(), f'Toy data not found at {TOY}. Run from the examples/ directory.'

## 1 Data generation

The toy dataset was generated with the `motco simulate` CLI command. You can regenerate it yourself if you have R and the InterSIM package installed:

```bash
# Install InterSIM in R (one-time)
Rscript -e 'install.packages("InterSIM", repos = c("https://cran.r-universe.dev", "https://cloud.r-project.org"))'

# Generate the toy dataset
motco simulate \
  --seed 42 \
  --n-samples 90 \
  --trajectory-mode orientation \
  --effect-size 1.0 \
  --prop-affected-features 0.1 \
  --cluster-mean-shift 0.10 \
  --out-dir data/toy/
```

The command writes:
- `methylation.csv`, `expression.csv`, `proteomics.csv` — raw omics matrices (samples × features)
- `metadata.csv` — sample metadata with `sample_id`, `group`, `stage`, `cluster`
- `model_full.csv`, `model_reduced.csv`, `ls_means.csv`, `contrast.json` — trajectory design
- `truth.json` — ground-truth injection parameters

The committed toy data uses moderate InterSIM cluster separation, so stage classification is intentionally informative but not perfect.

The pre-generated files are bundled in this repository so you can run the notebook without R.

In [ ]:
# Load the toy data
metadata = pd.read_csv(TOY / 'metadata.csv')
methylation = pd.read_csv(TOY / 'methylation.csv')
expression = pd.read_csv(TOY / 'expression.csv')
proteomics = pd.read_csv(TOY / 'proteomics.csv')
truth = json.loads((TOY / 'truth.json').read_text())

print(f'Samples: {len(metadata)}')
print(f'Groups:  {metadata["group"].value_counts().to_dict()}')
print(f'Stages:  {sorted(metadata["stage"].unique())}')
print(f'Omics dims: methylation={methylation.shape}, expression={expression.shape}, proteomics={proteomics.shape}')
print(f'Trajectory mode: {truth["trajectory_mode"]}, effect size: {truth["group_effect_size"]}')

## 2 PLS-DA latent space

**Supervised path.** We concatenate the three omics layers and run PLS-DA with `y = stage` to find a latent space that maximally co-varies with disease progression. The intuition: the resulting space is biologically meaningful for the disease trajectory, because disease information has "leaked" into it through the supervised procedure.

After double cross-validation selects the optimal number of latent variables, we fit a final model on all data and extract the X score matrix as the outcome matrix Y for trajectory analysis.

**Equivalent CLI command:**
```bash
motco plsr \
  --input data/toy/methylation.csv \
  --input data/toy/expression.csv \
  --input data/toy/proteomics.csv \
  --metadata data/toy/metadata.csv \
  --label-col stage \
  --cv1-splits 5 --cv2-splits 5 --n-repeats 10 \
  --out-table data/toy/plsr_table.csv \
  --out-scores data/toy/latent_pls.csv
```

In [ ]:
from motco.stats.pls import fit_plsda_transform, plsda_doubleCV

# Standardize each layer and concatenate
def standardize(df: pd.DataFrame) -> np.ndarray:
    values = df.to_numpy(dtype=float)
    mean = values.mean(axis=0, keepdims=True)
    std = values.std(axis=0, keepdims=True)
    std[std == 0.0] = 1.0
    return (values - mean) / std

X_concat = np.hstack([standardize(methylation), standardize(expression), standardize(proteomics)])
X_df = pd.DataFrame(X_concat)
y = metadata['stage']

print(f'Concatenated X shape: {X_df.shape}')
print(f'y (stage) unique values: {sorted(y.unique())}')

In [ ]:
N_REPEATS = int(os.getenv('MOTCO_NOTEBOOK_REPEATS', '3'))

cv_result = plsda_doubleCV(
    X=X_df, y=y,
    cv1_splits=3, cv2_splits=3, n_repeats=N_REPEATS,
    max_components=5,
)
print('CV table (best LV per repeat):')
print(cv_result['table'])

In [ ]:
# Modal LV count → fit final model → extract scores
modal_lv = int(cv_result['table'].iloc[:, 1].mode()[0])
print(f'Modal LV count from CV: {modal_lv}')

scores_pls = fit_plsda_transform(X_concat, y, n_components=modal_lv)
Y_pls = pd.DataFrame(scores_pls, columns=[f'lv_{i+1}' for i in range(modal_lv)],
                     index=metadata['sample_id'])
print(f'PLS latent space shape: {Y_pls.shape}')

## 3 SNF latent space

**Unsupervised path.** Similarity Network Fusion builds a patient similarity network from each omics layer separately, then iteratively fuses them. Spectral embedding of the fused network gives the latent coordinates.

Unlike PLS-DA, SNF is not supervised by disease stage — it captures structure across all omics layers jointly. Both paths converge at the same downstream trajectory analysis.

**Equivalent CLI command:**
```bash
motco snf \
  --input data/toy/methylation.csv \
  --input data/toy/expression.csv \
  --input data/toy/proteomics.csv \
  --K 10 --k 10 --t 20 \
  --out-embedding data/toy/latent_snf.csv
```

In [ ]:
from motco.stats.snf import SNF, get_affinity_matrix, get_spectral

n = len(metadata)
K = min(10, n - 1)

layers = [methylation.to_numpy(dtype=float),
          expression.to_numpy(dtype=float),
          proteomics.to_numpy(dtype=float)]

affinities = get_affinity_matrix(layers, K=K, eps=0.5)
fused = SNF(affinities, k=K, t=20)
embedding = get_spectral(fused, n_components=10)

Y_snf = pd.DataFrame(embedding,
                     columns=[f'snf_{i}' for i in range(embedding.shape[1])],
                     index=metadata['sample_id'])
print(f'SNF latent space shape: {Y_snf.shape}')

## 4 Trajectory design

The design files (model matrices, LS means, contrast) were pre-generated by `motco simulate` and are loaded directly. They encode the group × stage structure needed for trajectory analysis.

In [ ]:
model_full = pd.read_csv(TOY / 'model_full.csv').values
model_reduced = pd.read_csv(TOY / 'model_reduced.csv').values
ls_means = pd.read_csv(TOY / 'ls_means.csv').values
contrast = json.loads((TOY / 'contrast.json').read_text())

g_levels = sorted(metadata['group'].unique())
s_levels = sorted(metadata['stage'].unique())

print(f'Groups: {g_levels}, Stages: {s_levels}')
print(f'Full model matrix:    {model_full.shape}')
print(f'Reduced model matrix: {model_reduced.shape}')
print(f'LS means:             {ls_means.shape}  (one row per group×stage cell)')
print(f'Contrast:             {contrast}')

## 5 Differential trajectory analysis

`estimate_difference` computes three pairwise group statistics:
- **delta** — difference in trajectory magnitude (total path length)
- **angle** — angle in degrees between trajectory directions (0° = same, 90° = orthogonal)
- **shape** — Procrustes distance between trajectory shapes (requires ≥ 3 stages)

`RRPP` permutes residuals of the reduced model to produce null distributions and empirical p-values.

We run both on the PLS and SNF latent spaces to compare the two integration paths.

In [ ]:
from motco.stats import RRPP, estimate_difference

PERMS = int(os.getenv('MOTCO_NOTEBOOK_PERMS', '99'))
print(f'Using {PERMS} RRPP permutations (set MOTCO_NOTEBOOK_PERMS for more)')

def run_analysis(Y, label):
    d, a, sh = estimate_difference(Y.values, model_full, ls_means, contrast)
    dist_d, dist_a, dist_sh = RRPP(Y.values, model_full, model_reduced,
                                    ls_means, contrast, permutations=PERMS)
    def pval(null, obs, i, j):
        v = np.array([m[i, j] for m in null])
        return (np.sum(v >= obs) + 1) / (len(v) + 1)

    print(f'\n--- {label} ---')
    for i, g1 in enumerate(g_levels):
        for j, g2 in enumerate(g_levels):
            if j <= i:
                continue
            print(f'  {g1} vs {g2}:')
            print(f'    angle = {a[i,j]:.1f}°  (p = {pval(dist_a, a[i,j], i, j):.3f})')
            print(f'    delta = {d[i,j]:.4f}  (p = {pval(dist_d, d[i,j], i, j):.3f})')
            if len(s_levels) >= 3:
                print(f'    shape = {sh[i,j]:.4f}  (p = {pval(dist_sh, sh[i,j], i, j):.3f})')
    return d, a, sh

deltas_pls, angles_pls, shapes_pls = run_analysis(Y_pls, 'PLS-DA latent space')
deltas_snf, angles_snf, shapes_snf = run_analysis(Y_snf, 'SNF latent space')

## 6 Visualization

`plot_trajectory_from_data` fits a 2-D PCA on the latent space and draws each group's trajectory as a directed path through LS-mean positions. Arrows indicate the direction of change across stages.

In [ ]:
from motco.viz import plot_trajectory_from_data

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, Y, title in [
    (axes[0], Y_pls, 'PLS-DA latent space'),
    (axes[1], Y_snf, 'SNF latent space'),
]:
    plot_trajectory_from_data(
        Y=Y,
        metadata=metadata.set_index('sample_id'),
        group_col='group',
        level_col='stage',
        show_samples=True,
        ax=ax,
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()

---

**Summary:** Both PLS-DA and SNF detect the injected orientation difference (~90° angle between groups A and B). The p-values for the angle statistic should be small; the delta statistic (magnitude difference) reflects only the injected effect size on path length, which in orientation mode is zero by design — so delta p-values are expected to be non-significant.